# Agentic Data Engineering Workshop
## Modern ETL + Data Quality + AI Agent Readiness

เป้าหมาย: สร้าง deterministic ETL pipeline จากข้อมูลยอดขายและสภาพอากาศ แล้วสร้าง Quality Report ที่สามารถส่งต่อให้ n8n / AI Agent วิเคราะห์ต่อได้

## 0) เตรียมข้อมูล
ใน Colab ให้ Upload ไฟล์ในโฟลเดอร์ `data/` ได้แก่ `sales_raw.csv`, `product_master.csv`, `weather_daily.json` หรือแก้ `DATA_DIR` ให้ชี้ไปยังโฟลเดอร์ของคุณ

In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA_DIR = Path("/content/sample_data/data")  # แก้เป็น /content ถ้า upload ไฟล์ไว้ที่ root ของ Colab
OUT_DIR = Path("/content/sample_data/data")
OUT_DIR.mkdir(exist_ok=True)

## 1) Extract: อ่านข้อมูลจากหลายแหล่ง

In [ ]:
sales = pd.read_csv(DATA_DIR / "sales_raw.csv")
products = pd.read_csv(DATA_DIR / "product_master.csv")
weather = pd.read_json(DATA_DIR / "weather_daily.json")

print("sales", sales.shape)
print("products", products.shape)
print("weather", weather.shape)
display(sales)

sales (12, 7)
products (5, 3)
weather (4, 4)


,order_id,date,product,qty,price,channel,customer_note
0,O-1001,2026-08-28,Coffee,10.0,60,Store,regular order
1,O-1002,2026-08-28,Cake,5.0,80,Online,NaN
2,O-1001,2026-08-28,Coffee,10.0,60,Store,duplicate row
3,O-1003,2026/08/29,Tea,4.0,45,Store,different date format
4,O-1004,2026-08-29,Coffee,-2.0,60,Store,return entered as negative qty
5,O-1005,29 Aug 2026,Coffe,3.0,60,Online,product typo
6,O-1006,2026-08-29,Cookie,NaN,30,Online,missing qty
7,O-1007,2026-08-30,Sandwich,7.0,FREE,Store,free campaign price as text
8,O-1008,2026-08-30,Cake,500.0,80,Store,possible outlier qty
9,O-1009,2026-08-30,Tea,8.0,45,Online,normal


### Checkpoint 1
ตอบคำถาม: แถวใดมีปัญหาชัดเจน 3 อย่างแรกที่คุณเห็นคืออะไร?

In [ ]:
# Quick profile
sales.info()
sales.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       12 non-null     object 
 1   date           12 non-null     object 
 2   product        11 non-null     object 
 3   qty            11 non-null     float64
 4   price          12 non-null     object 
 5   channel        12 non-null     object 
 6   customer_note  11 non-null     object 
dtypes: float64(1), object(6)
memory usage: 804.0+ bytes


,order_id,date,product,qty,price,channel,customer_note
count,12,12,11,11.000000,12,12,11
unique,11,6,6,NaN,6,2,10
top,O-1001,2026-08-28,Coffee,NaN,60,Store,normal
freq,2,3,4,NaN,5,7,2
mean,NaN,NaN,NaN,50.272727,NaN,NaN,NaN
std,NaN,NaN,NaN,149.199927,NaN,NaN,NaN
min,NaN,NaN,NaN,-2.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,3.500000,NaN,NaN,NaN
50%,NaN,NaN,NaN,6.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,9.000000,NaN,NaN,NaN


## 2) Transform: Normalize + Approved Mapping
AI สามารถช่วยเสนอการแก้คำผิดได้ แต่ใน pipeline จริงต้องใช้ mapping ที่อนุมัติแล้วเท่านั้น

In [ ]:
PRODUCT_CORRECTIONS = {"Coffe": "Coffee"}

df = sales.copy()
df["date_raw"] = df["date"]
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date.astype("string")
df["product_raw"] = df["product"]
df["product"] = df["product"].replace(PRODUCT_CORRECTIONS)
df["qty_num"] = pd.to_numeric(df["qty"], errors="coerce")
df["price_num"] = pd.to_numeric(df["price"], errors="coerce")
display(df)

,order_id,date,product,qty,price,channel,customer_note,date_raw,product_raw,qty_num,price_num
0,O-1001,2026-08-28,Coffee,10.0,60,Store,regular order,2026-08-28,Coffee,10.0,60.0
1,O-1002,2026-08-28,Cake,5.0,80,Online,NaN,2026-08-28,Cake,5.0,80.0
2,O-1001,2026-08-28,Coffee,10.0,60,Store,duplicate row,2026-08-28,Coffee,10.0,60.0
3,O-1003,<NA>,Tea,4.0,45,Store,different date format,2026/08/29,Tea,4.0,45.0
4,O-1004,2026-08-29,Coffee,-2.0,60,Store,return entered as negative qty,2026-08-29,Coffee,-2.0,60.0
5,O-1005,<NA>,Coffee,3.0,60,Online,product typo,29 Aug 2026,Coffe,3.0,60.0
6,O-1006,2026-08-29,Cookie,NaN,30,Online,missing qty,2026-08-29,Cookie,NaN,30.0
7,O-1007,2026-08-30,Sandwich,7.0,FREE,Store,free campaign price as text,2026-08-30,Sandwich,7.0,NaN
8,O-1008,2026-08-30,Cake,500.0,80,Store,possible outlier qty,2026-08-30,Cake,500.0,80.0
9,O-1009,2026-08-30,Tea,8.0,45,Online,normal,2026-08-30,Tea,8.0,45.0


## 3) Validate: ใช้กฎที่ตรวจสอบซ้ำได้

In [ ]:

known_products = set(products["product"])
flag_cols = [] # สร้าง list เก็บผลลัพธ์

df["is_duplicate"] = df.duplicated(subset=["order_id"], keep="first")
df["invalid_date"] = df["date"].isna()
df["missing_product"] = df["product"].isna() | (df["product"].astype("string").str.strip() == "")
df["unknown_product"] = ~df["product"].isin(known_products) & ~df["missing_product"]
df["missing_qty"] = df["qty_num"].isna()
df["negative_qty"] = df["qty_num"] < 0
df["non_numeric_price"] = df["price_num"].isna()
df["outlier_qty"] = df["qty_num"] > 100

flag_cols = ["is_duplicate","invalid_date","missing_product","unknown_product","missing_qty","negative_qty","non_numeric_price","outlier_qty"]

# .any(axis=1) ตรวจว่าในแต่ละแถวมีค่า True อย่างน้อย 1 ตัวหรือไม่
df["is_valid"] = ~df[flag_cols].any(axis=1)

display(df[["order_id","date_raw","date","product_raw","product","qty","price"] + flag_cols + ["is_valid"]])

,order_id,date_raw,date,product_raw,product,qty,price,is_duplicate,invalid_date,missing_product,unknown_product,missing_qty,negative_qty,non_numeric_price,outlier_qty,is_valid
0,O-1001,2026-08-28,2026-08-28,Coffee,Coffee,10.0,60,False,False,False,False,False,False,False,False,True
1,O-1002,2026-08-28,2026-08-28,Cake,Cake,5.0,80,False,False,False,False,False,False,False,False,True
2,O-1001,2026-08-28,2026-08-28,Coffee,Coffee,10.0,60,True,False,False,False,False,False,False,False,False
3,O-1003,2026/08/29,<NA>,Tea,Tea,4.0,45,False,True,False,False,False,False,False,False,False
4,O-1004,2026-08-29,2026-08-29,Coffee,Coffee,-2.0,60,False,False,False,False,False,True,False,False,False
5,O-1005,29 Aug 2026,<NA>,Coffe,Coffee,3.0,60,False,True,False,False,False,False,False,False,False
6,O-1006,2026-08-29,2026-08-29,Cookie,Cookie,NaN,30,False,False,False,False,True,False,False,False,False
7,O-1007,2026-08-30,2026-08-30,Sandwich,Sandwich,7.0,FREE,False,False,False,False,False,False,True,False,False
8,O-1008,2026-08-30,2026-08-30,Cake,Cake,500.0,80,False,False,False,False,False,False,False,True,False
9,O-1009,2026-08-30,2026-08-30,Tea,Tea,8.0,45,False,False,False,False,False,False,False,False,True


### Checkpoint 2
เลือก 1 แถวที่ถูก reject และเขียนเหตุผลว่า reject เพราะกฎใด

## 4) Transform + Join
คำนวณ revenue และ join กับ Weather API ด้วย date

In [ ]:
valid = df[df["is_valid"]].copy() # กรองเฉพาะแถวในคอลัมน์ is_valid ที่มีค่า true
valid["revenue"] = valid["qty_num"] * valid["price_num"]

# แปลง date จาก text to date format
weather["date"] = pd.to_datetime(weather["date"]).dt.date.astype("string")

# ทำการ merge 3 ตารางเข้าด้วยกัน ิvalid, products, weather โดยตาราง valid คือตารางหลัก
clean = valid.merge(products, on="product", how="left").merge(weather, on="date", how="left")

# เลือกคอลัมน์ในข้อมูลที่ทำความสะอาดแล้ว
clean = clean[["order_id","date","product","qty_num","price_num","channel","category","revenue","temperature_c","rain_mm","condition"]]

# เปลี่ยนชื่อคอลัมน์ dty_num และ price_num
clean = clean.rename(columns={"qty_num":"qty","price_num":"price"})
display(clean)

,order_id,date,product,qty,price,channel,category,revenue,temperature_c,rain_mm,condition
0,O-1001,2026-08-28,Coffee,10.0,60.0,Store,Beverage,600.0,32,0,sunny
1,O-1002,2026-08-28,Cake,5.0,80.0,Online,Bakery,400.0,32,0,sunny
2,O-1009,2026-08-30,Tea,8.0,45.0,Online,Beverage,360.0,35,5,hot
3,O-1011,2026-08-31,Coffee,6.0,60.0,Online,Beverage,360.0,31,18,cloudy


## 5) Load: บันทึกผลลัพธ์

In [ ]:
# บันทึกไฟล์ใหม่เป็น clean_sales_weather.csv

clean.to_csv(OUT_DIR / "clean_sales_weather.csv", index=False)
df.to_csv(OUT_DIR / "sales_quality_flags.csv", index=False)
print("Saved:", OUT_DIR / "clean_sales_weather.csv")

Saved: /content/sample_data/data/clean_sales_weather.csv


## 6) Quality Report สำหรับ Agent

In [ ]:
# สรุปรายงานคุณภาพข้อมูล (Data Quality Report)

# สร้าง Dictionary ชื่อ report
report = {
    "pipeline": "daily_sales_weather",  # ระบุชื่อ Pipeline หรือ Workflow ที่กำลังประมวลผลอยู่
    "total_rows": int(len(df)), # นับแถวทั้งหมด
    "valid_rows": int(df["is_valid"].sum()), # นับจำนวนแถวที่เป็น True
    "rejected_rows": int((~df["is_valid"]).sum()), # นับจำนวนแถวที่ไม่ผ่านการตรวจสอบ เป็น False
}

# นับจำนวน Error แต่ละประเภท
for col in flag_cols:
    report[col] = int(df[col].sum())

# คำนวณ Severity / ทำไมใช้ max(total_rows,1) เพื่อป้องกันข้อมูลที่เป็น 0/0
report["severity"] = "HIGH" if report["rejected_rows"] / max(report["total_rows"], 1) > 0.2 else "LOW"

# กำหนดเงื่อนไขว่าต้องส่งให้คนตรวจหรือไม่
report["human_review_required"] = report["severity"] == "HIGH" or report["non_numeric_price"] > 0

# เชฟผลลัพธ์เป็น JSON
with open(OUT_DIR / "quality_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(json.dumps(report, ensure_ascii=False, indent=2))

{
  "pipeline": "daily_sales_weather",
  "total_rows": 12,
  "valid_rows": 4,
  "rejected_rows": 8,
  "is_duplicate": 1,
  "invalid_date": 2,
  "missing_product": 1,
  "unknown_product": 0,
  "missing_qty": 1,
  "negative_qty": 1,
  "non_numeric_price": 1,
  "outlier_qty": 1,
  "severity": "HIGH",
  "human_review_required": true
}


## 7) Prompt สำหรับ AI Agent
ใช้ผลจาก Quality Report เป็นหลักฐาน ไม่ให้ AI เดาข้อมูลเอง

In [ ]:
prompt = f"""
You are a Data Quality Analyst for an ETL pipeline.

Analyze this ETL quality report and explain whether the pipeline output is safe to publish.
Do not change production data. Do not invent missing facts.

Quality report:
{json.dumps(report, ensure_ascii=False, indent=2)}

Return only valid JSON with keys:
severity, plain_language_summary, likely_causes, recommended_actions, safe_to_publish, human_review_required, message_to_team.
"""
print(prompt)


You are a Data Quality Analyst for an ETL pipeline.

Analyze this ETL quality report and explain whether the pipeline output is safe to publish.
Do not change production data. Do not invent missing facts.

Quality report:
{
  "pipeline": "daily_sales_weather",
  "total_rows": 12,
  "valid_rows": 4,
  "rejected_rows": 8,
  "is_duplicate": 1,
  "invalid_date": 2,
  "missing_product": 1,
  "unknown_product": 0,
  "missing_qty": 1,
  "negative_qty": 1,
  "non_numeric_price": 1,
  "outlier_qty": 1,
  "severity": "HIGH",
  "human_review_required": true
}

Return only valid JSON with keys:
severity, plain_language_summary, likely_causes, recommended_actions, safe_to_publish, human_review_required, message_to_team.



## 8) Optional: เรียก LLM ผ่าน API เมื่อมี API Key
เซลล์นี้เป็นตัวอย่างเท่านั้น หากไม่มี API Key ให้ copy prompt ไปใช้ในเครื่องมือ AI ที่ผู้สอนกำหนด

In [ ]:
# Guardrail / Validation

# Optional example, not required for the workshop.
import os
from google import genai
from google.colab import userdata


if userdata.get('GEMINI_API_KEY'):
    client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
    response = client.models.generate_content(model="gemini-3.6-flash", contents=prompt)
    print(response.text)
else:
    print("No GEMINI_API_KEY found. Use the generated prompt manually.")

```json
{
  "severity": "HIGH",
  "plain_language_summary": "The daily_sales_weather pipeline processed 12 total rows, but only 4 rows (33.3%) were valid. 8 rows were rejected due to critical data quality issues, including duplicate records, invalid dates, missing/negative/outlier quantities, and a non-numeric price field.",
  "likely_causes": [
    "Upstream source file formatting changes or corrupt data exports causing invalid date formats and non-numeric price entries.",
    "Manual data entry errors leading to negative quantities, missing fields, and outlier quantity values.",
    "Upstream ingestion logic failing to deduplicate incoming records prior to processing."
  ],
  "recommended_actions": [
    "Do not publish the pipeline output to downstream production databases or reporting tools.",
    "Conduct a manual review of the 8 rejected records to identify and fix data anomalies.",
    "Contact the upstream data provider to investigate source-side data quality and schema complia

## Final reflection
1. ขั้นตอนใดควรใช้ deterministic rule?
2. ขั้นตอนใดให้ AI ช่วยได้?
3. ข้อมูลแบบใดต้องให้มนุษย์อนุมัติก่อนเผยแพร่?